# MapBiomas Fire Monitor — Benchmark /vsigs/ vs Download

**Objetivo**: Medir se a leitura de COGs via `/vsigs/` (streaming direto do GCS) e mais rapida e economica em RAM que o metodo atual (download → disco → leitura).

**Testes**:
| Teste | Metodo | O que mede |
|-------|--------|------------|
| A | Download → `rasterio.open(local)` | Tempo download + leitura + RAM cache disco |
| B | `/vsigs/` → `rasterio.open(vsi)` | Tempo leitura streaming + RAM |
| C | Download → `gdalbuildvrt` → `gdal_translate COG` | Simula merge M2 atual (NUM_THREADS=2) |
| D | `/vsigs/` → `gdalbuildvrt` → `gdal_translate COG` | Simula merge M2 otimizado (NUM_THREADS=ALL_CPUS) |

**Resultado esperado**: 3-5x mais rapido, zero RAM de cache de disco.

---
**Ambiente**: Colab ou Local (auto-detecta).  
**Dados reais**: bucket `mapbiomas-fire`, COGs do Peru (Sentinel-2 ou Landsat).

In [ ]:
# ============================================================
# CELL 1: SETUP — Paths, imports, deteccao de ambiente
# ============================================================
import sys, os, time, subprocess
import numpy as np

IS_COLAB = 'google.colab' in sys.modules
print(f'Ambiente: {"Google Colab" if IS_COLAB else "Local"}')

if IS_COLAB:
    if not os.path.exists('peru-fire'):
        print('Clonando repositorio...')
        subprocess.run(['git', 'clone', 'https://github.com/mapbiomas/peru-fire.git'], check=True)
    CORE_PATH = '/content/peru-fire/mapbiomas_fire_monitor/version_01/src/core'
    BENCH_PATH = '/content/peru-fire/tests/benchmarks'
else:
    cwd = os.getcwd()
    # Procura o pipeline a partir do CWD
    for candidate_root in [cwd, os.path.join(cwd, '..', '..')]:
        candidate = os.path.abspath(os.path.join(
            candidate_root, 'mapbiomas_fire_monitor', 'version_01', 'src', 'core'
        ))
        if os.path.exists(os.path.join(candidate, 'M0_auth_config.py')):
            CORE_PATH = candidate
            BENCH_PATH = os.path.abspath(os.path.join(candidate_root, 'tests', 'benchmarks'))
            break
    else:
        raise RuntimeError('Pipeline nao encontrado. Execute este notebook de tests/benchmarks/ ou da raiz do repo.')

if CORE_PATH not in sys.path:
    sys.path.insert(0, CORE_PATH)
if BENCH_PATH not in sys.path:
    sys.path.insert(0, BENCH_PATH)

print(f'CORE_PATH: {CORE_PATH}')
print(f'BENCH_PATH: {BENCH_PATH}')

# Importa o backend de benchmark
from vsigs_benchmark import (
    discover_periods, benchmark_read_download, benchmark_read_vsigs,
    benchmark_merge_download, benchmark_merge_vsigs,
    compare_arrays, get_ram_mb
)
print('Backend importado.')

In [ ]:
# ============================================================
# CELL 2: AUTENTICACAO — GEE + GCS
# ============================================================
from M0_auth_config import authenticate, CONFIG, _get_fs

print('Autenticando...')
authenticate()
print(f'Bucket: {CONFIG["bucket"]}')
print(f'Pais: {CONFIG["country"]}')
print(f'GCS prefix: {CONFIG["gcs_catalog_prefix"]}')

fs = _get_fs()
print('OK — autenticado.')

In [ ]:
# ============================================================
# CELL 3: DESCOBRIR PERIODOS DISPONIVEIS
# ============================================================
# Tenta Sentinel-2 primeiro, fallback Landsat
for sensor in ['sentinel2', 'landsat']:
    print(f'\n--- Buscando COGs {sensor.upper()} ---')
    periods = discover_periods(sensor=sensor)
    if periods:
        SENSOR = sensor
        break

if not periods:
    print('\n[ERRO] Nenhum COG encontrado. Verifique o bucket e as credenciais.')
else:
    print(f'\nSensor: {SENSOR}')
    print(f'Periodos encontrados: {len(periods)}')
    for p, bands in periods.items():
        n = len(bands)
        band_list = ', '.join(sorted(bands.keys()))
        print(f'  {p}  ({n} bandas: {band_list})')

In [ ]:
# ============================================================
# CELL 4: CONFIGURAR PERIODO DE TESTE
# ============================================================
# Altere estas variaveis para testar um periodo especifico.
# Deixe None para auto-selecionar o primeiro com >= 3 bandas.

SELECTED_PERIOD = None  # Ex: '2025_08'

if SELECTED_PERIOD is None:
    for p, bands in periods.items():
        if len(bands) >= 3:
            SELECTED_PERIOD = p
            break

if SELECTED_PERIOD is None:
    SELECTED_PERIOD = list(periods.keys())[0]

selected_bands = periods[SELECTED_PERIOD]
print(f'Periodo selecionado: {SELECTED_PERIOD}')
print(f'Bandas disponiveis ({len(selected_bands)}): {sorted(selected_bands.keys())}')

# Pega a primeira banda para os testes A e B (leitura individual)
TEST_BAND = sorted(selected_bands.keys())[0]
TEST_URI = selected_bands[TEST_BAND]
print(f'\nBanda para teste A/B: {TEST_BAND}')
print(f'Path: {TEST_URI}')

---
## Teste A: Download → rasterio

Metodo atual: baixa o COG do GCS para disco local, depois le com rasterio.

In [ ]:
# ============================================================
# CELL 5: TESTE A — Download + rasterio
# ============================================================
print(f'Teste A: Download + rasterio')
print(f'  Arquivo: {os.path.basename(TEST_URI)}')
print(f'  Tamanho estimado: calculando...')

try:
    metrics_a, arr_a = benchmark_read_download(TEST_URI, fs=fs)
    print(f'\n  Resultados:')
    print(f'    Download:      {metrics_a["time_download_s"]}s')
    print(f'    Leitura:        {metrics_a["time_read_s"]}s')
    print(f'    Total:          {metrics_a["time_download_s"] + metrics_a["time_read_s"]}s')
    print(f'    RAM delta (DL): +{metrics_a["ram_delta_download_mb"]} MB')
    print(f'    RAM delta (RD): +{metrics_a["ram_delta_read_mb"]} MB')
    print(f'    Tamanho arquivo:{metrics_a["file_size_mb"]} MB')
    print(f'    Shape:          {metrics_a["shape"]}')
    print(f'    Dtype:          {metrics_a["dtype"]}')
    print(f'    Pixels validos: {metrics_a["pixels_valid"]:,} / {metrics_a["pixels_total"]:,} ({metrics_a["pct_valid"]}%)')
    print(f'    Min/Max/Mean:   {metrics_a["min"]} / {metrics_a["max"]} / {metrics_a["mean"]}')
except Exception as e:
    print(f'\n  [ERRO] Teste A falhou: {e}')
    metrics_a = None
    arr_a = None

## Teste B: /vsigs/ → rasterio

Metodo streaming: le o COG direto do GCS via `/vsigs/`, sem download para disco.

In [ ]:
# ============================================================
# CELL 6: TESTE B — /vsigs/ + rasterio
# ============================================================
print(f'Teste B: /vsigs/ + rasterio')
print(f'  Arquivo: {os.path.basename(TEST_URI)}')

try:
    metrics_b, arr_b = benchmark_read_vsigs(TEST_URI)
    print(f'\n  Resultados:')
    print(f'    Leitura:        {metrics_b["time_read_s"]}s')
    print(f'    RAM delta:      +{metrics_b["ram_delta_mb"]} MB')
    print(f'    Shape:          {metrics_b["shape"]}')
    print(f'    Dtype:          {metrics_b["dtype"]}')
    print(f'    Pixels validos: {metrics_b["pixels_valid"]:,} / {metrics_b["pixels_total"]:,} ({metrics_b["pct_valid"]}%)')
    print(f'    Min/Max/Mean:   {metrics_b["min"]} / {metrics_b["max"]} / {metrics_b["mean"]}')
except Exception as e:
    print(f'\n  [ERRO] Teste B falhou: {e}')
    metrics_b = None
    arr_b = None

## Comparacao pixel a pixel: Teste A vs Teste B

In [ ]:
# ============================================================
# CELL 7: COMPARACAO — Download vs /vsigs/ (pixel a pixel)
# ============================================================
if arr_a is not None and arr_b is not None:
    comp = compare_arrays(arr_a, arr_b)
    print('Comparacao pixel a pixel:')
    print(f'  Mesmo shape:  {comp["same_shape"]}')
    print(f'  Mesmo dtype:  {comp["same_dtype"]}')
    print(f'  Identicos:    {comp["identical"]}')
    if not comp['identical']:
        print(f'  Max diff:     {comp["max_diff"]}')
        print(f'  Mean diff:    {comp["mean_diff"]}')
        print(f'  % diferentes: {comp["pct_different"]}%')
    
    if metrics_a and metrics_b:
        dl_total = metrics_a['time_download_s'] + metrics_a['time_read_s']
        vs_total = metrics_b['time_read_s']
        speedup = round(dl_total / vs_total, 1) if vs_total > 0 else float('inf')
        print(f'\n  Speedup /vsigs/: {speedup}x mais rapido')
        print(f'  RAM economizada (cache disco): ~{metrics_a["ram_delta_download_mb"]:.0f} MB')
else:
    print('[ERRO] Arrays nao disponiveis para comparacao.')

---
## Teste C: Simulacao M2 merge — Metodo ATUAL (download)

Simula o que `M2_mosaic_logic.assemble_country_mosaic()` faz: baixa N bandas, gdalbuildvrt, gdal_translate COG com NUM_THREADS=2.

In [ ]:
# ============================================================
# CELL 8: TESTE C — M2 merge (download)
# ============================================================
# Seleciona ate 3 bandas modelo (red, nir, swir1) ou as primeiras disponiveis
model_bands = ['red', 'nir', 'swir1']
merge_bands = {}
for b in model_bands:
    if b in selected_bands:
        merge_bands[b] = selected_bands[b]

if len(merge_bands) < 2:
    available = sorted(selected_bands.keys())
    for b in available[:3]:
        if b not in merge_bands:
            merge_bands[b] = selected_bands[b]

print(f'Teste C: M2 merge (download)')
print(f'  {len(merge_bands)} bandas: {", ".join(merge_bands.keys())}')
print(f'  NUM_THREADS: 2 (atual)')

try:
    merge_uris = list(merge_bands.values())
    metrics_c, arr_c = benchmark_merge_download(merge_uris, fs=fs, num_threads=2)
    print(f'\n  Resultados:')
    print(f'    Download:      {metrics_c["time_download_s"]}s')
    print(f'    gdalbuildvrt:  {metrics_c["time_vrt_s"]}s')
    print(f'    gdal_translate: {metrics_c["time_translate_s"]}s')
    print(f'    TOTAL:          {metrics_c["total_time_s"]}s')
    print(f'    COG saida:     {metrics_c["output_size_mb"]} MB')
except Exception as e:
    print(f'\n  [ERRO] Teste C falhou: {e}')
    metrics_c = None
    arr_c = None

## Teste D: Simulacao M2 merge — Metodo OTIMIZADO (/vsigs/)

Simula o merge com `/vsigs/`: gdalbuildvrt direto do GCS (sem download), gdal_translate COG com NUM_THREADS=ALL_CPUS.

In [ ]:
# ============================================================
# CELL 9: TESTE D — M2 merge (/vsigs/)
# ============================================================
import os as _os
nw = _os.cpu_count() or 4

print(f'Teste D: M2 merge (/vsigs/)')
print(f'  {len(merge_bands)} bandas: {", ".join(merge_bands.keys())}')
print(f'  NUM_THREADS: {nw} (ALL_CPUS)')

try:
    metrics_d, arr_d = benchmark_merge_vsigs(merge_uris, num_threads=nw)
    if 'gdal_vsigs_error' in metrics_d:
        print(f'\n  [AVISO] GDAL /vsigs/ NAO SUPORTADO neste ambiente!')
        print(f'  Motivo: {metrics_d["gdal_vsigs_error"]}')
        print(f'  \n  Isso e ESPERADO no Colab. Use rasterio /vsigs/ como fallback.')
        print(f'  O M5 ja usa rasterio /vsigs/ com sucesso.')
    else:
        print(f'\n  Resultados:')
        print(f'    gdalbuildvrt:  {metrics_d["time_vrt_s"]}s (sem download)')
        print(f'    gdal_translate: {metrics_d["time_translate_s"]}s')
        print(f'    TOTAL:          {metrics_d["total_time_s"]}s')
        print(f'    COG saida:     {metrics_d["output_size_mb"]} MB')
except Exception as e:
    print(f'\n  [ERRO] Teste D falhou: {e}')
    metrics_d = None
    arr_d = None

## Comparacao pixel a pixel: Teste C vs Teste D

In [ ]:
# ============================================================
# CELL 10: COMPARACAO — M2 download vs M2 /vsigs/
# ============================================================
if arr_c is not None and arr_d is not None:
    comp_m2 = compare_arrays(arr_c, arr_d, label1='M2 download', label2='M2 /vsigs/')
    print('Comparacao Merge M2 pixel a pixel:')
    print(f'  Mesmo shape:  {comp_m2["same_shape"]}')
    print(f'  Mesmo dtype:  {comp_m2["same_dtype"]}')
    print(f'  Identicos:    {comp_m2["identical"]}')
    if not comp_m2['identical']:
        print(f'  Max diff:     {comp_m2["max_diff"]}')
        print(f'  Mean diff:    {comp_m2["mean_diff"]}')
        print(f'  % diferentes: {comp_m2["pct_different"]}%')
elif arr_d is None:
    print('Comparacao indisponivel: GDAL /vsigs/ nao funcionou neste ambiente.')
    print('Isso e esperado no Colab. O merge M2 no Colab deve manter download + paralelismo (ThreadPoolExecutor).')
else:
    print('Comparacao indisponivel: arrays nao disponiveis.')

---
## Resumo Final

In [ ]:
# ============================================================
# CELL 11: TABELA RESUMO
# ============================================================
print('=' * 70)
print('RESUMO — Benchmark /vsigs/ vs Download')
print(f'Data: {time.strftime("%Y-%m-%d %H:%M")}')
print(f'Ambiente: {"Colab" if IS_COLAB else "Local"}')
print(f'Periodo: {SELECTED_PERIOD} | Sensor: {SENSOR}')
print('=' * 70)

# Leitura individual
if metrics_a and metrics_b:
    dl_total = metrics_a['time_download_s'] + metrics_a['time_read_s']
    vs_total = metrics_b['time_read_s']
    print(f'\n--- Leitura de 1 COG ({TEST_BAND}) ---')
    print(f'  Download + read:  {dl_total:.1f}s  | RAM cache: +{metrics_a["ram_delta_download_mb"]:.0f} MB')
    print(f'  /vsigs/ read:     {vs_total:.1f}s  | RAM cache: 0 MB')
    if vs_total > 0:
        print(f'  Speedup:          {dl_total/vs_total:.1f}x')

# Merge M2
if metrics_c and metrics_d:
    print(f'\n--- Merge M2 ({len(merge_bands)} bandas) ---')
    print(f'  Download + VRT + COG (THREADS=2):  {metrics_c["total_time_s"]:.1f}s')
    if 'gdal_vsigs_error' in metrics_d:
        print(f'  /vsigs/ VRT + COG:                 ERRO — GDAL sem GCS')
        print(f'  > Fallback Colab: usar download + ThreadPoolExecutor')
    else:
        print(f'  /vsigs/ VRT + COG (THREADS={metrics_d.get("num_threads","?")}):{metrics_d["total_time_s"]:.1f}s')
        if metrics_d['total_time_s'] > 0:
            print(f'  Speedup:          {metrics_c["total_time_s"]/metrics_d["total_time_s"]:.1f}x')

# Pixel integrity
if arr_a is not None and arr_b is not None:
    comp = compare_arrays(arr_a, arr_b)
    print(f'\n--- Integridade dos dados ---')
    print(f'  Download vs /vsigs/ pixels identicos: {comp["identical"]}')

print(f'\n--- Conclusao ---')
if IS_COLAB:
    print(f'  COLUNA 1 (rapida): rasterio /vsigs/ para leitura')
    print(f'  COLUNA 1 (rapida): ThreadPoolExecutor para bandas/stats')
    print(f'  COLUNA 2 (media):  GDAL /vsigs/ NAO funciona, manter download + paralelismo')
else:
    print(f'  COLUNA 1+2: /vsigs/ + ThreadPoolExecutor em tudo')

print('=' * 70)
print('Fim do benchmark.')